In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from src.to_html import cell, heading, paragraph, table

In [2]:
data_dir = Path("../monthly_processed_data")
files = sorted(data_dir.glob("data-*.parquet"))[-2:]

df_list = []
for file in files:
    df_list.append(pd.read_parquet(file, columns=["delay_in_min", "train_name", "train_type", "is_canceled"]))
df = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(df):,} records from {len(files)} files")
print(f"Files: {[f.name for f in files]}")

Loaded 29,304,268 records from 2 files
Files: ['data-2026-01.parquet', 'data-2026-02.parquet']


In [3]:
# Define long-distance train types
long_distance_train_types = ["ICE", "IC", "FLX", "EC"]

# Calculate average delays, cancellation percentages, and sample counts by train
train_stats = (
    df[df["train_type"].isin(long_distance_train_types)]
    .groupby("train_name")
    .agg({"delay_in_min": ["mean", "count"], "is_canceled": "mean"})
    .sort_values(("delay_in_min", "count"), ascending=False)
)

# Flatten column names and rename for clarity
train_stats.columns = ["Durchschnittliche Versp\u00e4tung [min]", "Anzahl Fahrten", "Ausfallquote [%]"]
train_stats = train_stats.reset_index()
train_stats = train_stats.rename(columns={"train_name": "Zugname"})

# Convert cancellation rate to percentage
train_stats["Ausfallquote [%]"] = (train_stats["Ausfallquote [%]"] * 100).round(2)
train_stats["Durchschnittliche Versp\u00e4tung [min]"] = train_stats["Durchschnittliche Versp\u00e4tung [min]"].round(2)

# Filter to trains with at least 100 planned journeys
train_stats = train_stats[train_stats["Anzahl Fahrten"] >= 100]

# Reorder columns
train_stats = train_stats[["Zugname", "Durchschnittliche Versp\u00e4tung [min]", "Ausfallquote [%]", "Anzahl Fahrten"]]

print(f"Found {len(train_stats)} long-distance trains with at least 100 journeys")

Found 884 long-distance trains with at least 100 journeys


In [4]:
content = cell(
    heading("Welche Fernverkehrszüge haben am meisten Verspätungen?", level=1)
    + paragraph(
        "Viele Fernverkehrszüge haben eine eindeutige Nummer in den Daten und damit lässt sich die "
        "Pünktlichkeit und Ausfallquote aller Fernverbindungen von ICE, IC, FLX (FlixTrain) oder EC (EuroCity) ermitteln. "
        "Die Tabelle zeigt alle Züge mit mindestens 100 geplanten Fahrten."
    )
    + table(train_stats)
)
display(HTML(content))

Zugname ⇕,Durchschnittliche Verspätung [min] ⇕,Ausfallquote [%] ⇕,Anzahl Fahrten ⇕
FLX 10,24.14,4.68,4615
FLX 30,18.97,4.98,3030
IC 2034,13.53,18.53,1322
IC 2035,6.8,23.75,1305
IC 2338,10.86,22.57,1298
IC 2039,7.2,23.41,1290
IC 2435,9.43,15.31,1287
IC 2037,6.58,22.33,1254
IC 2036,16.27,18.46,1246
ICE 619,16.07,5.68,1198


In [5]:
start_date = files[0].stem.replace("data-", "")
end_date = files[-1].stem.replace("data-", "")
content = cell(
    f'<p style="font-size: 0.8em; text-align: center;">Quelle: <a href="https://github.com/piebro/deutsche-bahn-data/blob/main/notebooks/zugverbindung.ipynb">Berechnet</a> auf Basis von <a href="https://github.com/piebro/deutsche-bahn-data">gesammelten Daten</a> von der Deutschen Bahn vom {start_date} bis {end_date}.</p>'
)
display(HTML(content))